# NLP Zero To Hero Part 2: Sentences Sequencing

* Taking sentences and converting them into sequences of numbers.

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/svenwu/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
sentences = [
  'I love my dog', 
  'I love my cat', 
  'You love my dog!', 
  'Do you think my dog is amazing?'
]

In [5]:
tokenizer = Tokenizer(num_words=100)
tokenizer.fit_on_texts(sentences)
word_index = tokenizer.word_index

# "texts_to_sequences" method creates sequences of tokens representing each sentence.
# Notice in the output, how the dictionary values of each word is mapped to the sequences generated.
# {'my': 1, 'love': 2, 'dog': 3, 'i': 4,...} --> "I love my dog" --> [4, 2, 1, 3] 
sequences = tokenizer.texts_to_sequences(sentences)

print(word_index)
print(sequences)

{'my': 1, 'love': 2, 'dog': 3, 'i': 4, 'you': 5, 'cat': 6, 'do': 7, 'think': 8, 'is': 9, 'amazing': 10}
[[4, 2, 1, 3], [4, 2, 1, 6], [5, 2, 1, 3], [7, 5, 8, 1, 3, 9, 10]]


* Handling words in text that a neural network has never seen before.

In [9]:
# Sequencing sentences containing words NOT in the word index, like "manatee" and "really".
test_data = [
  'i really love my dog', 
  'my dog loves my manatee'
]

# 'i really love my dog' becomes --> 'i love my dog' --> [4, 2, 1, 3]
# 'my dog loves my manatee' becomes --> 'my dog my' --> [1, 3, 1]
 
test_seq = tokenizer.texts_to_sequences(test_data)
print(test_seq)

[[4, 2, 1, 3], [1, 3, 1]]


In [11]:
# Using the "oov_token" token property and setting it to "<OOV>", for words tokenzier does not recognize with the "Out of Vocabulary" token instead.
# Helps us maintain the sequence length as the SAME LENGTH as the sentence.
sentences = [
  'I love my dog', 
  'I love my cat', 
  'You love my dog!', 
  'Do you think my dog is amazing?'
]

tokenizer = Tokenizer(num_words=100, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)
word_index = tokenizer.word_index

sequences = tokenizer.texts_to_sequences(sentences)

test_data = [
  'i really love my dog', 
  'my dog loves my manatee'
]

test_seq = tokenizer.texts_to_sequences(test_data)
print(test_seq)

[[5, 1, 3, 2, 4], [2, 4, 1, 2, 1]]


### Handling Sentences of Different Lengths

* RaggedTensor

* Padding

In [12]:
# Import "pad_sequences" for PADDING.
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [25]:
sentences = [
  'I love my dog', 
  'I love my cat', 
  'You love my dog!', 
  'Do you think my dog is amazing?'
]

tokenizer = Tokenizer(num_words=100, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)
word_index = tokenizer.word_index

sequences = tokenizer.texts_to_sequences(sentences)

# Pad our sequences.
# 'I love my dog' --> [5, 2, 3, 4] then we pad it --> [0, 0, 0, 5, 3, 2, 4].
# It has 3 ADDITIONAL '0's at the start, because the LONGEST sentence has 7 words/
# This ensures all sentences has EQUALLY SIZED sequences.
# '0' means padding
padded = pad_sequences(sequences)

print(word_index)
print(sequences)
print(padded)

{'<OOV>': 1, 'my': 2, 'love': 3, 'dog': 4, 'i': 5, 'you': 6, 'cat': 7, 'do': 8, 'think': 9, 'is': 10, 'amazing': 11}
[[5, 3, 2, 4], [5, 3, 2, 7], [6, 3, 2, 4], [8, 6, 9, 2, 4, 10, 11]]
[[ 0  0  0  5  3  2  4]
 [ 0  0  0  5  3  2  7]
 [ 0  0  0  6  3  2  4]
 [ 8  6  9  2  4 10 11]]


### Putting Padding at END

* Putting the 0 padding at the end

In [22]:
sentences = [
  'I love my dog', 
  'I love my cat', 
  'You love my dog!', 
  'Do you think my dog is amazing?'
]

tokenizer = Tokenizer(num_words=100, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)
word_index = tokenizer.word_index

sequences = tokenizer.texts_to_sequences(sentences)

# Pad our sequences.
# 'I love my dog' --> [5, 2, 3, 4] then we pad it --> [0, 0, 0, 5, 3, 2, 4].
# It has 3 ADDITIONAL '0's at the start, because the LONGEST sentence has 7 words/
# This ensures all sentences has EQUALLY SIZED sequences.
# '0' means padding
# padded = pad_sequences(sequences)

# Putting padding at the end
padded = pad_sequences(sequences, padding='post')

print(word_index)
print(sequences)
print(padded)

{'<OOV>': 1, 'my': 2, 'love': 3, 'dog': 4, 'i': 5, 'you': 6, 'cat': 7, 'do': 8, 'think': 9, 'is': 10, 'amazing': 11}
[[5, 3, 2, 4], [5, 3, 2, 7], [6, 3, 2, 4], [8, 6, 9, 2, 4, 10, 11]]
[[ 5  3  2  4  0  0  0]
 [ 5  3  2  7  0  0  0]
 [ 6  3  2  4  0  0  0]
 [ 8  6  9  2  4 10 11]]


### Setting Desired Length

* Can specify DESIRED LENGTH, if you don't want the length of the padded sentences to be the same as the longest sentence.

In [23]:
sentences = [
  'I love my dog', 
  'I love my cat', 
  'You love my dog!', 
  'Do you think my dog is amazing?'
]

tokenizer = Tokenizer(num_words=100, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)
word_index = tokenizer.word_index

sequences = tokenizer.texts_to_sequences(sentences)

# Pad our sequences.
# 'I love my dog' --> [5, 2, 3, 4] then we pad it --> [0, 0, 0, 5, 3, 2, 4].
# It has 3 ADDITIONAL '0's at the start, because the LONGEST sentence has 7 words/
# This ensures all sentences has EQUALLY SIZED sequences.
# '0' means padding
# padded = pad_sequences(sequences)

# Can specify DESIRED LENGTH, if you don't want the length of the padded sentences to be the same as the longest sentence.
padded = pad_sequences(sequences, padding='post', maxlen=5)

print(word_index)
print(sequences)
print(padded)

{'<OOV>': 1, 'my': 2, 'love': 3, 'dog': 4, 'i': 5, 'you': 6, 'cat': 7, 'do': 8, 'think': 9, 'is': 10, 'amazing': 11}
[[5, 3, 2, 4], [5, 3, 2, 7], [6, 3, 2, 4], [8, 6, 9, 2, 4, 10, 11]]
[[ 5  3  2  4  0]
 [ 5  3  2  7  0]
 [ 6  3  2  4  0]
 [ 9  2  4 10 11]]


### Truncating the Sequence

* # What if sentences are LONGER than the specified maxlen? Decide how to truncate with 'post' or 'pre'.

In [24]:
sentences = [
  'I love my dog', 
  'I love my cat', 
  'You love my dog!', 
  'Do you think my dog is amazing?'
]

tokenizer = Tokenizer(num_words=100, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)
word_index = tokenizer.word_index

sequences = tokenizer.texts_to_sequences(sentences)

# Pad our sequences.
# 'I love my dog' --> [5, 2, 3, 4] then we pad it --> [0, 0, 0, 5, 3, 2, 4].
# It has 3 ADDITIONAL '0's at the start, because the LONGEST sentence has 7 words/
# This ensures all sentences has EQUALLY SIZED sequences.
# '0' means padding
# padded = pad_sequences(sequences)

# What if sentences are LONGER than the specified maxlen? Decide how to truncate with 'post' or 'pre'
padded = pad_sequences(sequences, padding='post', truncating='post', maxlen=5)

print(word_index)
print(sequences)
print(padded)

{'<OOV>': 1, 'my': 2, 'love': 3, 'dog': 4, 'i': 5, 'you': 6, 'cat': 7, 'do': 8, 'think': 9, 'is': 10, 'amazing': 11}
[[5, 3, 2, 4], [5, 3, 2, 7], [6, 3, 2, 4], [8, 6, 9, 2, 4, 10, 11]]
[[5 3 2 4 0]
 [5 3 2 7 0]
 [6 3 2 4 0]
 [8 6 9 2 4]]
